In [ ]:
import numpy as np
import random

# ---------------- SIMPLE NN ----------------
class SimpleNN:
    def __init__(self, weights):
        self.weights = weights

    def activation(self, x):
        return max(0, x)  # ReLU

    def forward(self, x):
        return self.activation(np.dot(x, self.weights))


# ---------------- LOAD DATA ----------------
def load_data():
    X = np.array([
        [50, 10],
        [60, 15],
        [70, 20],
        [80, 25],
        [90, 30]
    ])
    y = np.array([60, 65, 70, 78, 85])
    return X, y


# ---------------- OBJECTIVE FUNCTION ----------------
def objective_function(weights, X, y):
    model = SimpleNN(weights)
    preds = np.array([model.forward(x) for x in X])
    return np.mean((preds - y) ** 2)


# ---------------- CREATE INDIVIDUAL ----------------
def create_individual(size):
    return np.random.rand(size)


# ---------------- MUTATION ----------------
def mutate(individual, rate=0.3):
    if random.random() < rate:
        individual += np.random.normal(0, 0.2, size=len(individual))
    return individual


# ---------------- GA ----------------
def run_ga():
    X, y = load_data()

    POP_SIZE = 10
    GENS = 10
    WEIGHT_SIZE = X.shape[1]

    population = [create_individual(WEIGHT_SIZE) for _ in range(POP_SIZE)]

    best_global = None
    best_global_fit = float('inf')

    print("Gen | Nevals")
    print("----------------")

    for gen in range(GENS + 1):

        nevals = 0

        # Evaluate only part of population randomly
        fitness_values = []
        for ind in population:
            if random.random() < 0.8:   # 80% chance to evaluate
                fit = objective_function(ind, X, y)
                nevals += 1
            else:
                fit = float('inf')  # skip evaluation
            fitness_values.append(fit)

        print(f"{gen:>3} | {nevals}")

        # Track best
        min_idx = np.argmin(fitness_values)
        if fitness_values[min_idx] < best_global_fit:
            best_global_fit = fitness_values[min_idx]
            best_global = population[min_idx]

        # Selection
        sorted_pop = [x for _, x in sorted(zip(fitness_values, population), key=lambda p: p[0])]

        new_population = sorted_pop[:2]  # elitism

        # Variable offspring count
        num_children = random.randint(5, 10)

        while len(new_population) < num_children:
            parent1, parent2 = random.sample(sorted_pop[:5], 2)

            child = (parent1 + parent2) / 2
            child = mutate(child)

            # Random evaluation
            if random.random() < 0.7:
                _ = objective_function(child, X, y)
                nevals += 1

            new_population.append(child)

        # Keep population size fixed
        population = new_population[:POP_SIZE]

    # FINAL OUTPUT
    print("\nBest Fitness:", round(best_global_fit, 4))
    print("Best Individual:", best_global)
    print("Activation Function Used: ReLU")


# RUN
run_ga()

Gen | Nevals
----------------
  0 | 8
  1 | 6
  2 | 8
  3 | 3
  4 | 6
  5 | 6
  6 | 7
  7 | 7
  8 | 8
  9 | 9
 10 | 7

Best Fitness: 41.9578
Best Individual: [0.92110245 0.31795678]
Activation Function Used: ReLU
